# SplatStream Lab — Reproducible CUDA Evaluation

This notebook reproduces the real-scene validation path for SplatStream Lab using **Mip-NeRF 360 / Bonsai** and a pinned `gsplat` revision.

Stage 1 records the environment, trains a 7,000-step 3D Gaussian Splatting baseline, exports the checkpoint to PLY, and runs the portable compression sanity check. Stage 2 reuses that checkpoint for a **full-scene held-out CUDA rate-distortion sweep**.

**Runtime requirement:** select a GPU runtime before execution and keep the same runtime connected through both stages.


## 1. Train and validate the 7K Bonsai baseline


In [ ]:
from pathlib import Path
import urllib.request

runner = Path('/content/run_bonsai_cuda.py')
url = 'https://raw.githubusercontent.com/reusahn/splatstream-lab/main/tools/run_bonsai_cuda.py'
urllib.request.urlretrieve(url, runner)
print('Runner:', runner)

%run /content/run_bonsai_cuda.py


## 2. Full-scene CUDA rate-distortion sweep

This stage verifies the pinned source tree, Bonsai COLMAP data, and 7,000-step checkpoint produced above. It then launches the complete rate-distortion sweep in **one fresh Python interpreter** so `gsplat` torch-library kernels are registered exactly once.

The evaluator applies the same view-independent importance heuristic to the full scene, creates four pruning/quantization operating points, and evaluates every point on the same held-out Bonsai cameras with the full `gsplat` rasterizer.

If the Colab runtime has been reset and `/content` was cleared, rerun Stage 1 before Stage 2.


In [ ]:
from pathlib import Path
import urllib.request

launcher = Path('/content/run_full_cuda_pipeline_v3.py')
url = 'https://raw.githubusercontent.com/reusahn/splatstream-lab/main/tools/run_full_cuda_pipeline_v3.py'
urllib.request.urlretrieve(url, launcher)
print('Full CUDA launcher:', launcher)

%run /content/run_full_cuda_pipeline_v3.py


## Outputs

Stage 1 produces **`splatstream_bonsai_evidence.zip`**. Stage 2 produces **`splatstream_full_cuda_rd_v2.zip`**, containing the full rate-distortion CSV/JSON, method notes, environment metadata, plots, and held-out validation renders.

Measured values should be reported together with the GPU model, pinned commit IDs, dataset source, evaluation split, and payload definition.
